In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [8]:
import numpy as np
import pandas as pd
import os
import warnings
import joblib
import gc

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

import lightgbm as lgb
import mlflow
import mlflow.sklearn
import dagshub

warnings.filterwarnings("ignore")
print("All imports OK ✓")

All imports OK ✓


# **dagshub**

In [9]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "LightGBM_Training"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name}")
else:
    print(f"Found existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print("MLflow + DagsHub connected ✓")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a3c563a2-61e9-4729-9fa1-d810e23b7ff2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=edf9e5d2167a0eba4d5f08cbbf441151803ae1e0db215057ff0ecc61095d4795




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

Created experiment: LightGBM_Training
MLflow + DagsHub connected ✓


# Load & merge

In [10]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading data...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")
print(f"  train_transaction: {train_tr.shape}")
print(f"  train_identity:    {train_id.shape}")
print(f"  test_transaction:  {test_tr.shape}")
print(f"  test_identity:     {test_id.shape}")

print("\nFixing column names (hyphen → underscore)...")
train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

print("\nMerging...")
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

print("\nAligning columns...")
only_in_train = set(X.columns) - set(test.columns)
only_in_test  = set(test.columns) - set(X.columns)
print(f"  Cols only in train: {len(only_in_train)}")
print(f"  Cols only in test:  {len(only_in_test)}")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"  Aligned: X={X.shape}, test={test.shape}")
print(f"  Fraud rate: {y.mean():.4f} ({y.sum()} fraud out of {len(y)} total)")

Loading data...
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  test_transaction:  (506691, 393)
  test_identity:     (141907, 41)

Fixing column names (hyphen → underscore)...

Merging...
  train merged: (590540, 434)
  test merged:  (506691, 433)

Aligning columns...
  Cols only in train: 0
  Cols only in test:  0
  Aligned: X=(590540, 432), test=(506691, 432)
  Fraud rate: 0.0350 (20663 fraud out of 590540 total)


#  Feature Engineering

In [13]:
print("=== FEATURE ENGINEERING ===")
cols_before_engineering = X.shape[1]

def engineer_features(df, is_train=True, card1_stats=None):
    # 1. TransactionDT → day and time features ──
    df["TransactionDay"]  = (df["TransactionDT"] / 86400).astype(np.float32)
    df["TransactionHour"] = ((df["TransactionDT"] / 3600) % 24).astype(np.float32)
    df["TransactionWeekday"] = (df["TransactionDay"] % 7).astype(np.float32)

    # 2. D column normalization 
    # D1 = days since card began → D1n = day card began (almost constant per client)
    df["D1n"] = df["TransactionDay"] - df["D1"]
    df["D2n"] = df["TransactionDay"] - df["D2"]
    df["D3n"] = df["TransactionDay"] - df["D3"]
    df["D4n"] = df["TransactionDay"] - df["D4"]
    df["D10n"] = df["TransactionDay"] - df["D10"]
    df["D15n"] = df["TransactionDay"] - df["D15"]

    # ── 3. TransactionAmt features ──
    df["TransactionAmt_log"]      = np.log1p(df["TransactionAmt"]).astype(np.float32)
    df["TransactionAmt_is_round"] = (df["TransactionAmt"] % 1 == 0).astype(np.int8)
    df["TransactionAmt_cents"]    = (df["TransactionAmt"] % 1).astype(np.float32)

    # ── 4. email provider ──
    if "P_emaildomain" in df.columns:
        df["P_email_provider"] = df["P_emaildomain"].str.split(".").str[0].fillna("unknown")
    if "R_emaildomain" in df.columns:
        df["R_email_provider"] = df["R_emaildomain"].str.split(".").str[0].fillna("unknown")

    # ── 5. C columns aggregates ──
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if len(c_cols) > 0:
        df["C_mean"] = df[c_cols].mean(axis=1).astype(np.float32)
        df["C_std"]  = df[c_cols].std(axis=1).astype(np.float32)
        df["C_max"]  = df[c_cols].max(axis=1).astype(np.float32)

    return df

# ── UID feature ──
# combine card1 + addr1 + D1n to identify unique clients
print("Creating UID features...")

# compute D1n for train to build aggregates
X["TransactionDay"] = (X["TransactionDT"] / 86400).astype(np.float32)
X["D1n"] = X["TransactionDay"] - X["D1"]

test["TransactionDay"] = (test["TransactionDT"] / 86400).astype(np.float32)
test["D1n"] = test["TransactionDay"] - test["D1"]

# create uid = card1 + addr1 + D1n (rounded to nearest day)
X["uid"]    = X["card1"].astype(str) + "_" + X["addr1"].astype(str) + "_" + X["D1n"].fillna(-1).round(0).astype(int).astype(str)
test["uid"] = test["card1"].astype(str) + "_" + test["addr1"].astype(str) + "_" + test["D1n"].fillna(-1).round(0).astype(int).astype(str)
print(f"  Unique UIDs in train: {X['uid'].nunique()}")
print(f"  Unique UIDs in test:  {test['uid'].nunique()}")

# ── aggregate only C columns by UID (M cols are strings) ──
print("Creating UID aggregated features...")
c_cols = [c for c in X.columns if c.startswith("C") and c[1:].isdigit()]
print(f"  Aggregating {len(c_cols)} C columns by UID...")

uid_agg = X.groupby("uid")[c_cols].agg(["mean", "std"]).reset_index()
uid_agg.columns = ["uid"] + [f"uid_{c[0]}_{c[1]}" for c in uid_agg.columns[1:]]

X    = X.merge(uid_agg, on="uid", how="left")
test = test.merge(uid_agg, on="uid", how="left")
print(f"  After UID agg: X={X.shape}, test={test.shape}")

# ── card1 aggregates (computed from train only) ──
print("Computing card1 aggregates...")
card1_count      = X["card1"].value_counts()
card1_amt_median = X.groupby("card1")["TransactionAmt"].median()
card1_amt_std    = X.groupby("card1")["TransactionAmt"].std()

X["card1_count"]      = X["card1"].map(card1_count).astype(np.float32)
X["card1_amt_median"] = X["card1"].map(card1_amt_median).astype(np.float32)
X["card1_amt_std"]    = X["card1"].map(card1_amt_std).astype(np.float32)

test["card1_count"]      = test["card1"].map(card1_count).astype(np.float32)
test["card1_amt_median"] = test["card1"].map(card1_amt_median).astype(np.float32)
test["card1_amt_std"]    = test["card1"].map(card1_amt_std).astype(np.float32)

# ── apply remaining feature engineering ──
print("Engineering remaining features...")
engineer_features(X)
engineer_features(test)
gc.collect()

# ── drop uid — don't use directly  ──
X    = X.drop(columns=["uid"])
test = test.drop(columns=["uid"])

# realign
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)

cols_after_engineering = X.shape[1]
print(f"\nFinal: X={X.shape}, test={test.shape}")
print(f"New cols added: {cols_after_engineering - cols_before_engineering}")
print("Feature engineering done ✓")

=== FEATURE ENGINEERING ===
Creating UID features...
  Unique UIDs in train: 243655
  Unique UIDs in test:  218267
Creating UID aggregated features...
  Aggregating 14 C columns by UID...
  After UID agg: X=(590540, 463), test=(506691, 463)
Computing card1 aggregates...
Engineering remaining features...

Final: X=(590540, 480), test=(506691, 480)
New cols added: 45
Feature engineering done ✓


# feature selection

In [14]:
print("=== FEATURE SELECTION ===")
print(f"Starting: X={X.shape}")

# ── 1. drop high missing >90% ──
print("\nStep 1: Dropping >90% missing...")
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"  Dropped: {len(high_missing)} cols")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  After: X={X.shape}")

# ── 2. drop near-zero variance ──
print("\nStep 2: Dropping near-zero variance (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"  Dropped: {len(low_var)} cols → {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  After: X={X.shape}")

# ── 3. drop raw C columns (aggregated into uid features already) ──
print("\nStep 3: Dropping raw C columns...")
c_cols_dropped = [c for c in X.columns if c.startswith("C") and c[1:].isdigit()]
print(f"  Dropping: {c_cols_dropped}")
X    = X.drop(columns=c_cols_dropped)
test = test.drop(columns=c_cols_dropped)
print(f"  After: X={X.shape}")

# ── 4. drop high corr (less aggressive — 0.99) ──
print("\nStep 4: Dropping highly correlated (r > 0.99)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = X[num_cols_temp].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.99)]
print(f"  Dropped: {len(high_corr)} cols")
X    = X.drop(columns=high_corr)
test = test.drop(columns=high_corr)
print(f"  After: X={X.shape}")

# ── 5. drop high cardinality (>200 unique) ──
print("\nStep 5: Dropping high cardinality (>200 unique)...")
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"  Dropped: {len(high_card)} → {high_card}")
X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)
print(f"  After: X={X.shape}")

assert list(X.columns) == list(test.columns)
cols_after_cleaning = X.shape[1]
final_feature_count = X.shape[1]
final_cols          = X.columns.tolist()
print(f"\n=== FINAL: {X.shape} ===")
print("Columns aligned ✓")

=== FEATURE SELECTION ===
Starting: X=(590540, 480)

Step 1: Dropping >90% missing...
  Dropped: 12 cols
  After: X=(590540, 468)

Step 2: Dropping near-zero variance (std < 0.01)...
  Dropped: 2 cols → ['V1', 'V305']
  After: X=(590540, 466)

Step 3: Dropping raw C columns...
  Dropping: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  After: X=(590540, 452)

Step 4: Dropping highly correlated (r > 0.99)...
  Dropped: 48 cols
  After: X=(590540, 404)

Step 5: Dropping high cardinality (>200 unique)...
  Dropped: 2 → ['id_33', 'DeviceInfo']
  After: X=(590540, 402)

=== FINAL: (590540, 402) ===
Columns aligned ✓


# MEMORY REDUCTION

In [15]:
print("=== MEMORY REDUCTION ===")
def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")
print(f"\nColumn dtypes:")
print(X.dtypes.value_counts())

=== MEMORY REDUCTION ===
  X: 1520.6 MB
  test: 1315.1 MB

Column dtypes:
float32    370
object      29
int32        1
int16        1
int8         1
Name: count, dtype: int64


# TRAIN VALIDATION SPLIT (BASED ON TIME)

In [16]:
print("=== TIME-BASED SPLIT (1st place approach) ===")
print(f"TransactionDT range: {X['TransactionDT'].min()} to {X['TransactionDT'].max()}")

# sort by time first
X = X.sort_values("TransactionDT")
y = y[X.index]
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

# split: train on first 80% of time, validate on last 20%
split_point = X["TransactionDT"].quantile(0.8)
print(f"Split point (80th percentile): {split_point:.0f}")

train_mask = X["TransactionDT"] <= split_point
val_mask   = X["TransactionDT"] >  split_point

X_train = X[train_mask].reset_index(drop=True)
X_val   = X[val_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_val   = y[val_mask].reset_index(drop=True)

print(f"\n  X_train: {X_train.shape}, fraud rate: {y_train.mean():.4f}")
print(f"  X_val:   {X_val.shape},   fraud rate: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"\n  Numeric cols:     {len(num_cols)}")
print(f"  Categorical cols: {len(cat_cols)}")
print(f"  Cat cols: {cat_cols}")

del X
gc.collect()
print("\nFreed X from memory ✓")

=== TIME-BASED SPLIT (1st place approach) ===
TransactionDT range: 86400 to 15811131
Split point (80th percentile): 12192854

  X_train: (472432, 402), fraud rate: 0.0351
  X_val:   (118108, 402),   fraud rate: 0.0344

  Numeric cols:     373
  Categorical cols: 29
  Cat cols: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'P_email_provider', 'R_email_provider']

Freed X from memory ✓


# TRAIN

In [17]:
from sklearn.model_selection import RandomizedSearchCV

print("=== LIGHTGBM TRAINING WITH HYPERPARAMETER SEARCH ===")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

fraud_count      = int(y_train.sum())
nonfraud_count   = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count
print(f"  scale_pos_weight: {scale_pos_weight:.2f}")

base_lgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", lgb.LGBMClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

param_dist = {
    "classifier__n_estimators":     [200, 300, 500],
    "classifier__max_depth":        [4, 6, 8, -1],
    "classifier__learning_rate":    [0.01, 0.02, 0.05, 0.1],
    "classifier__subsample":        [0.6, 0.7, 0.8, 0.9],
    "classifier__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "classifier__min_child_samples":[20, 50, 100, 200],
    "classifier__reg_alpha":        [0, 0.01, 0.1, 0.5],
    "classifier__reg_lambda":       [0.5, 1.0, 2.0, 5.0],
    "classifier__num_leaves":       [31, 63, 127, 255],
}

print(f"\nRunning 10 random combinations with 3-fold CV...")
print("This may take 20-40 minutes...")

search = RandomizedSearchCV(
    base_lgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=1
)

search.fit(X_train, y_train)
print(f"\nBest CV AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

model_lgb = search.best_estimator_
print("Best model selected ✓")

# ── evaluate ──
print("\n=== EVALUATING ===")
val_probs = model_lgb.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"  AUC:       {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

# ── show all trials ──
print("\n=== ALL TRIALS ===")
results_df = pd.DataFrame(search.cv_results_).sort_values("mean_test_score", ascending=False)
for _, row in results_df.iterrows():
    p = {k.replace("classifier__", ""): v for k, v in row["params"].items()}
    print(f"  Rank {int(row['rank_test_score'])}: AUC={row['mean_test_score']:.4f} ±{row['std_test_score']:.4f} | {p}")

# ── log cleaning ──
print("\nLogging to MLflow...")
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_param("missing_threshold",   0.9)
    mlflow.log_param("variance_threshold",  0.01)
    mlflow.log_param("corr_threshold",      0.99)
    mlflow.log_param("high_card_threshold", 200)
    mlflow.log_param("cols_after_cleaning", cols_after_cleaning)
    mlflow.log_param("dropped_high_missing", len(high_missing))
    mlflow.log_param("dropped_low_variance", len(low_var))
    mlflow.log_param("dropped_high_corr",    len(high_corr))
    mlflow.log_param("dropped_high_card",    len(high_card))
    with open("lgb_dropped_columns.txt", "w") as f:
        f.write("HIGH MISSING:\n" + "\n".join(high_missing))
        f.write("\n\nLOW VARIANCE:\n" + "\n".join(low_var))
        f.write("\n\nHIGH CORR:\n" + "\n".join(high_corr))
        f.write("\n\nHIGH CARDINALITY:\n" + "\n".join(high_card))
    mlflow.log_artifact("lgb_dropped_columns.txt")
    print(f"  LightGBM_Cleaning Run ID: {mlflow.active_run().info.run_id}")

# ── log feature engineering ──
with mlflow.start_run(run_name="LightGBM_Feature_Engineering"):
    mlflow.log_param("cols_before_engineering", cols_before_engineering)
    mlflow.log_param("cols_after_engineering",  cols_after_engineering)
    mlflow.log_param("uid_feature_created",     True)
    mlflow.log_param("uid_agg_CM_cols",         True)
    mlflow.log_param("D_cols_normalized",       True)
    mlflow.log_param("TransactionAmt_log",      True)
    mlflow.log_param("card1_aggregates",        True)
    mlflow.log_param("time_based_split",        True)
    print(f"  LightGBM_Feature_Engineering Run ID: {mlflow.active_run().info.run_id}")

# ── log feature selection ──
with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    mlflow.log_param("final_feature_count", final_feature_count)
    with open("lgb_final_features.txt", "w") as f:
        f.write("\n".join(final_cols))
    mlflow.log_artifact("lgb_final_features.txt")
    print(f"  LightGBM_Feature_Selection Run ID: {mlflow.active_run().info.run_id}")

# ── log each trial ──
results_df2 = pd.DataFrame(search.cv_results_)
for i, row in results_df2.iterrows():
    trial_params = {k.replace("classifier__", ""): v for k, v in row["params"].items()}
    trial_params["scale_pos_weight"] = round(scale_pos_weight, 2)
    with mlflow.start_run(run_name=f"LightGBM_Trial_{i+1}"):
        mlflow.log_params(trial_params)
        mlflow.log_metrics({
            "cv_auc_mean": round(row["mean_test_score"], 4),
            "cv_auc_std":  round(row["std_test_score"],  4),
            "cv_rank":     int(row["rank_test_score"]),
        })
        print(f"  Trial {i+1} logged — AUC: {row['mean_test_score']:.4f}")

# ── log best model ──
best_params = {k.replace("classifier__", ""): v for k, v in search.best_params_.items()}
best_params["scale_pos_weight"] = round(scale_pos_weight, 2)
best_params["validation_strategy"] = "time_based_80_20"

with mlflow.start_run(run_name="LightGBM_Training_Best"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        "val_auc":       val_auc,
        "val_precision": val_precision,
        "val_recall":    val_recall,
        "val_f1":        val_f1,
        "cv_auc":        round(search.best_score_, 4),
    })
    mlflow.sklearn.log_model(
        model_lgb,
        artifact_path="lightgbm_pipeline",
        registered_model_name="LightGBM_Fraud_Pipeline"
    )
    print(f"  LightGBM_Training_Best Run ID: {mlflow.active_run().info.run_id}")

print("\nAll runs logged to DagsHub ✓")

del X_train
del X_val
gc.collect()
print("Freed memory ✓")

=== LIGHTGBM TRAINING WITH HYPERPARAMETER SEARCH ===
  scale_pos_weight: 27.46

Running 10 random combinations with 3-fold CV...
This may take 20-40 minutes...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.02, classifier__max_depth=-1, classifier__min_child_samples=100, classifier__n_estimators=500, classifier__num_leaves=63, classifier__reg_alpha=0.1, classifier__reg_lambda=1.0, classifier__subsample=0.8; total time= 1.3min
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.02, classifier__max_depth=-1, classifier__min_child_samples=100, classifier__n_estimators=500, classifier__num_leaves=63, classifier__reg_alpha=0.1, classifier__reg_lambda=1.0, classifier__subsample=0.8; total time= 1.3min
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.02, classifier__max_depth=-1, classifier__min_child_samples=100, classifier__n_estimators=500, classifier__num_leaves=63

2026/05/03 14:22:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:23:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LightGBM_Fraud_Pipeline'.
2026/05/03 14:23:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_Fraud_Pipeline, version 1
Created version '1' of model 'LightGBM_Fraud_Pipeline'.


  LightGBM_Training_Best Run ID: 04e5c625293e443189017729c5463273
🏃 View run LightGBM_Training_Best at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/3/runs/04e5c625293e443189017729c5463273
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/3

All runs logged to DagsHub ✓
Freed memory ✓


In [18]:
print("=== GENERATING SUBMISSION ===")
print(f"  test shape: {test.shape}")

test_probs = model_lgb.predict_proba(test)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Predicted fraud rate: {test_probs.mean():.4f}")
print(f"  Min: {test_probs.min():.4f}, Max: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv("submission_lightgbm.csv", index=False)
print("\nsubmission_lightgbm.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION ===
  test shape: (506691, 402)
  Predictions shape: (506691,)
  Predicted fraud rate: 0.2644
  Min: 0.0286, Max: 0.9530

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.112687
1        3663550  0.306334
2        3663551  0.390691
3        3663552  0.131105
4        3663553  0.263735
5        3663554  0.102384
6        3663555  0.430811
7        3663556  0.426366
8        3663557  0.126058
9        3663558  0.216908

submission_lightgbm.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
